# 02_hrv_calibration_transfer.ipynb

Calibration transfer: test classifiers trained on WESAD against Empatica and Fitbit HRV features.

**Goal:** Compare performance before vs. after calibration.

In [11]:
import pandas as pd
from pathlib import Path
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
import matplotlib.pyplot as plt

# ----------------------
# Resolve project root
# ----------------------
project_root = Path.cwd().resolve()
while project_root.name.lower() == "notebooks":
    project_root = project_root.parent

data_raw = project_root / "data" / "raw"
data_processed = project_root / "data" / "processed"
results_dir = project_root / "results"
results_dir.mkdir(exist_ok=True)

print("Project root:", project_root)
print("Data processed path:", data_processed)

Project root: C:\Users\Sunil\Projects\HRV-GenAI
Data processed path: C:\Users\Sunil\Projects\HRV-GenAI\data\processed


In [12]:
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay, RocCurveDisplay

# Ensure manuscript/figures folder exists
figures_dir = project_root / "manuscript" / "figures"
figures_dir.mkdir(parents=True, exist_ok=True)


In [13]:
# =====================================================
# Load WESAD ECG HRV Features (from parquet files)
# =====================================================
import pandas as pd
from pathlib import Path

wesad_dir = project_root / "data" / "processed" / "wesad"

dfs = []
for f in wesad_dir.glob("*_ecg.parquet"):
    df = pd.read_parquet(f)
    df["Subject"] = f.stem.split("_")[0]   # Add subject ID like "S2"
    dfs.append(df)

wesad = pd.concat(dfs, ignore_index=True)

print("✅ Combined WESAD ECG features:", wesad.shape)
display(wesad.head())


✅ Combined WESAD ECG features: (60807600, 3)


,time_sec,ECG,Subject
0,0.000000,-1.333694,S10
1,0.001429,-1.327744,S10
2,0.002857,-1.322067,S10
3,0.004286,-1.316345,S10
4,0.005714,-1.310257,S10


In [14]:
# =========================
# 2. Train on WESAD
# =========================
# Detect HRV feature columns (case-insensitive)
expected_features = ["meanhr", "sdnn", "rmssd", "lf", "hf", "lf_hf"]
col_map = {c.lower(): c for c in wesad.columns}

features = [col_map[f] for f in expected_features if f in col_map]
print("Using features:", features)

# Detect label column
label_candidates = [c for c in wesad.columns if "label" in c.lower()]
if not label_candidates:
    raise ValueError("No label column found in WESAD data!")
label_col = label_candidates[0]
print("Using label column:", label_col)

# Assign training data
X_w = wesad[features]
y_w = wesad[label_col]

# Scale features
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_w)

print("✅ WESAD data prepared:", X_scaled.shape, y_w.shape)


Using features: []


ValueError: No label column found in WESAD data!

In [ ]:
# =====================================================
# 2. Transfer Evaluation (using baseline_with_labels.csv)
# =====================================================

import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

# -----------------------------------------------------
# 2.1 Load baseline predictions + labels
# -----------------------------------------------------
df = pd.read_csv(results_dir / "baseline_with_labels.csv")

print("✅ Loaded baseline_with_labels.csv")
print("Columns:", df.columns.tolist())
display(df.head())

# Extract arrays
y_true       = df["y_true"].values
y_pred_uncal = df["y_pred"].values   # uncalibrated predictions
y_pred_cal   = df["y_prob"].values   # calibrated predictions (probabilities)

# -----------------------------------------------------
# 2.2 Define evaluation functions
# -----------------------------------------------------
def evaluate_model(y_true, y_pred):
    """Compute standard metrics for binary classification."""
    return {
        "Accuracy": accuracy_score(y_true, np.round(y_pred)),
        "Precision": precision_score(y_true, np.round(y_pred), zero_division=0),
        "Recall": recall_score(y_true, np.round(y_pred), zero_division=0),
        "F1-score": f1_score(y_true, np.round(y_pred), zero_division=0),
        "ROC AUC": roc_auc_score(y_true, y_pred)
    }

def evaluate_transfer_performance(y_true, y_pred_uncal, y_pred_cal):
    results = []
    results.append({"Dataset": "Uncalibrated", **evaluate_model(y_true, y_pred_uncal)})
    results.append({"Dataset": "Calibrated",   **evaluate_model(y_true, y_pred_cal)})

    table4 = pd.DataFrame(results)
    print("=== Table 4: Calibration and Transfer Learning Performance ===")
    print(table4.to_string(index=False))
    return table4

# -----------------------------------------------------
# 2.3 Run evaluation and save
# -----------------------------------------------------
table4 = evaluate_transfer_performance(y_true, y_pred_uncal, y_pred_cal)

out_file = results_dir / "wesad_transfer_results_eval.csv"
table4.to_csv(out_file, index=False)
print(f"✅ Evaluated transfer results saved to: {out_file}")


In [ ]:
# =========================
# 4. Calibration Step
# =========================
feature_cols = ["MeanHR", "SDNN", "RMSSD", "LF", "HF", "LF_HF"]

calib_scaler = StandardScaler()
all_data = pd.concat([
    wesad[feature_cols],
    empatica[feature_cols],
    fitbit[feature_cols]
], ignore_index=True)

calib_scaler.fit(all_data)

results.append(evaluate_transfer(empatica, "Empatica (Calibrated)", calib_scaler, clf))
results.append(evaluate_transfer(fitbit, "Fitbit (Calibrated)", calib_scaler, clf))

results_df = pd.DataFrame(results)
display(results_df)


In [ ]:
# =====================================================
# 2. Transfer Calibration + Evaluation
# =====================================================

import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

# -----------------------------------------------------
# 2.1 Load transfer predictions + labels
# -----------------------------------------------------
import pandas as pd

# Point to the same results folder under project root
results_dir = project_root / "results"

df = pd.read_csv(results_dir / "baseline_with_labels.csv")

print("✅ Loaded baseline_with_labels.csv")
print("Columns:", df.columns.tolist())
display(df.head())

# Extract arrays
y_true       = df["y_true"].values
y_pred_uncal = df["y_pred"].values   # discrete class predictions (uncalibrated)
y_pred_cal   = df["y_prob"].values   # probabilities (calibrated)

# -----------------------------------------------------
# 2.2 Define evaluation functions
# -----------------------------------------------------
def evaluate_model(y_true, y_pred):
    """Compute standard metrics for binary classification."""
    return {
        "Accuracy": accuracy_score(y_true, np.round(y_pred)),
        "Precision": precision_score(y_true, np.round(y_pred), zero_division=0),
        "Recall": recall_score(y_true, np.round(y_pred), zero_division=0),
        "F1-score": f1_score(y_true, np.round(y_pred), zero_division=0),
        "ROC AUC": roc_auc_score(y_true, y_pred)
    }

def evaluate_transfer_performance(y_true, y_pred_uncal, y_pred_cal):
    results = []
    results.append({"Dataset": "Uncalibrated", **evaluate_model(y_true, y_pred_uncal)})
    results.append({"Dataset": "Calibrated",   **evaluate_model(y_true, y_pred_cal)})

    table4 = pd.DataFrame(results)
    print("=== Table 4: Calibration and Transfer Learning Performance ===")
    print(table4.to_string(index=False))
    return table4

# -----------------------------------------------------
# 2.3 Run evaluation + save Table 4
# -----------------------------------------------------
table4 = evaluate_transfer_performance(y_true, y_pred_uncal, y_pred_cal)

out_file = results_dir / "wesad_transfer_results_eval.csv"
table4.to_csv(out_file, index=False)

print(f"✅ Evaluated transfer results saved to {out_file}")
display(table4)


In [ ]:
# =====================================================
# Figure 2. HRV Signal Alignment Across Modalities
# =====================================================

plt.figure(figsize=(8,5))

# Plot first 200 samples for clarity
plt.plot(y_true[:200], label="ECG (Ground Truth)", alpha=0.8)
plt.plot(y_pred_uncal[:200], label="PPG (Uncalibrated)", alpha=0.7)
plt.plot(y_pred_cal[:200], label="PPG (Calibrated)", alpha=0.7)

plt.title("Figure 2. HRV Signal Alignment Across Modalities")
plt.xlabel("Sample Index")
plt.ylabel("Prediction / Probability")
plt.legend()
plt.tight_layout()
plt.savefig(figures_dir / "Figure2_SignalAlignment.png", dpi=300)
plt.show()


In [ ]:
# =====================================================
# Figure 3. HRV Feature Distributions
# =====================================================
import seaborn as sns

# Combine features from ECG (wesad) and PPG (empatica, fitbit)
wesad_features = wesad[feature_cols].copy()
wesad_features["Source"] = "ECG (WESAD)"

empatica_features = empatica[feature_cols].copy()
empatica_features["Source"] = "PPG (Empatica)"

fitbit_features = fitbit[feature_cols].copy()
fitbit_features["Source"] = "PPG (Fitbit)"

# Merge into one DataFrame
features_df = pd.concat([wesad_features, empatica_features, fitbit_features], ignore_index=True)

# Plot distributions for selected HRV features
plt.figure(figsize=(12,6))
sns.violinplot(x="Source", y="RMSSD", data=features_df, inner="box", cut=0)
plt.title("Figure 3. HRV Feature Distributions (RMSSD)")
plt.tight_layout()
plt.savefig(figures_dir / "Figure3_FeatureDistributions_RMSSD.png", dpi=300)
plt.show()

# Optional: repeat for LF/HF and Sample Entropy (if available)
plt.figure(figsize=(12,6))
sns.violinplot(x="Source", y="LF_HF", data=features_df, inner="box", cut=0)
plt.title("Figure 3. HRV Feature Distributions (LF/HF)")
plt.tight_layout()
plt.savefig(figures_dir / "Figure3_FeatureDistributions_LFHF.png", dpi=300)
plt.show()


In [ ]:
# =====================================================
# Figure 4B. Confusion Matrix – Calibrated PPG Model
# =====================================================
from sklearn.metrics import ConfusionMatrixDisplay

plt.figure(figsize=(5,5))
ConfusionMatrixDisplay.from_predictions(y_true, np.round(y_pred_cal), cmap="Blues")
plt.title("Figure 4B. Confusion Matrix – Calibrated PPG Model")
plt.tight_layout()
plt.savefig(figures_dir / "Figure4B_ConfusionMatrix_Calibrated.png", dpi=300)
plt.show()


In [ ]:
# =====================================================
# Save Table 4 to manuscript/tables
# =====================================================
tables_dir = project_root / "manuscript" / "tables"
tables_dir.mkdir(parents=True, exist_ok=True)

table4_path = tables_dir / "Table4_TransferCalibration.csv"
table4.to_csv(table4_path, index=False)

print(f"✅ Table 4 (Transfer Calibration Results) saved to: {table4_path}")
display(table4)


In [ ]:
import pandas as pd
from sklearn.metrics import RocCurveDisplay
import matplotlib.pyplot as plt

# Path to calibration file
calibration_file = project_root / "data" / "processed" / "Empatica_Fitbit_Daily_HRVCalibration.csv"

# Load calibration results
df = pd.read_csv(calibration_file)

# Inspect to confirm columns
print(df.head())

# Assuming the file has columns like: y_true, y_pred_uncal, y_pred_cal
plt.figure(figsize=(6,6))
RocCurveDisplay.from_predictions(df["y_true"], df["y_pred_uncal"], name="PPG Uncalibrated", ax=plt.gca())
RocCurveDisplay.from_predictions(df["y_true"], df["y_pred_cal"], name="PPG Calibrated", ax=plt.gca())
plt.plot([0,1],[0,1],"k--", alpha=0.6)
plt.title("Figure 5A. ROC Curves – PPG Model (Empatica + Fitbit)")
plt.tight_layout()

out_file = project_root / "manuscript" / "figures" / "Figure5A_ROC_PPG.png"
plt.savefig(out_file, dpi=300)
plt.show()

print("Columns:", df.columns.tolist())
print(df.head())
print(f"✅ ROC curve saved to {out_file}")


In [ ]:

# =====================================================
# Save Debug Predictions for Empatica & Fitbit
# =====================================================
import pandas as pd
import os

results_dir = project_root / "results"
os.makedirs(results_dir, exist_ok=True)

# Extract ground truth labels
y_true_empatica = empatica["label"].values
y_true_fitbit   = fitbit["label"].values

# Get uncalibrated + calibrated predictions
y_pred_empatica_uncal = model.predict_proba(X_empatica)[:,1]
y_pred_fitbit_uncal   = model.predict_proba(X_fitbit)[:,1]

y_pred_empatica_cal = calib_model.predict_proba(X_empatica)[:,1]
y_pred_fitbit_cal   = calib_model.predict_proba(X_fitbit)[:,1]

# Save debug CSVs
pd.DataFrame({
    "y_true": y_true_empatica,
    "y_pred_uncal": y_pred_empatica_uncal,
    "y_pred_cal": y_pred_empatica_cal
}).to_csv(results_dir / "empatica_transfer_debug.csv", index=False)

pd.DataFrame({
    "y_true": y_true_fitbit,
    "y_pred_uncal": y_pred_fitbit_uncal,
    "y_pred_cal": y_pred_fitbit_cal
}).to_csv(results_dir / "fitbit_transfer_debug.csv", index=False)

print("✅ Saved raw predictions for Empatica and Fitbit to results/")


In [ ]:

# =====================================================
# Figure 5A. Combined ROC Curves – ECG, PPG, LSTM, CNN
# =====================================================
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import RocCurveDisplay

plt.figure(figsize=(7,7))

# ECG baseline (from Notebook 01)
df_base = pd.read_csv(project_root / "results" / "baseline_with_labels.csv")
RocCurveDisplay.from_predictions(df_base["y_true"], df_base["y_prob"], name="ECG Baseline", ax=plt.gca())

# Empatica transfer results
df_emp = pd.read_csv(project_root / "results" / "empatica_transfer_debug.csv")
RocCurveDisplay.from_predictions(df_emp["y_true"], df_emp["y_pred_uncal"], name="Empatica Uncalibrated", ax=plt.gca())
RocCurveDisplay.from_predictions(df_emp["y_true"], df_emp["y_pred_cal"], name="Empatica Calibrated", ax=plt.gca())

# Fitbit transfer results
df_fit = pd.read_csv(project_root / "results" / "fitbit_transfer_debug.csv")
RocCurveDisplay.from_predictions(df_fit["y_true"], df_fit["y_pred_uncal"], name="Fitbit Uncalibrated", ax=plt.gca())
RocCurveDisplay.from_predictions(df_fit["y_true"], df_fit["y_pred_cal"], name="Fitbit Calibrated", ax=plt.gca())

# Sequence model results (Notebook 03 extended eval)
try:
    df_seq = pd.read_csv(project_root / "results" / "wesad_sequence_results_eval_extended.csv")
    if "y_prob_lstm" in df_seq.columns:
        RocCurveDisplay.from_predictions(df_seq["y_true"], df_seq["y_prob_lstm"], name="LSTM", ax=plt.gca())
    if "y_prob_cnn" in df_seq.columns:
        RocCurveDisplay.from_predictions(df_seq["y_true"], df_seq["y_prob_cnn"], name="CNN", ax=plt.gca())
except Exception as e:
    print("⚠️ Could not load sequence model ROC:", e)

# Plot diagonal reference
plt.plot([0,1],[0,1],"k--", alpha=0.6)

plt.title("Figure 5A. Combined ROC Curves – ECG, PPG, LSTM, CNN")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.legend(loc="lower right")
plt.tight_layout()

out_file = project_root / "manuscript" / "figures" / "Figure5A_ROC_Combined.png"
plt.savefig(out_file, dpi=300)
plt.show()

print(f"✅ Combined ROC saved to {out_file}")
